# 03 · Modeling — HDBSCAN + classifier

**Author: Chris Kim** — 2026-05, **ChainSense** · UCSD DS3

Clustering and labeling were run on Google Colab (A100 GPU), not this machine. The code cells below are pasted from that Colab notebook; see the note on why the hardware is required.


## Why this ran on an A100

HDBSCAN doesn't assign points to centroids like k-means — it reconstructs the density structure of the whole dataset: pairwise distances → a mutual-reachability graph → a minimum spanning tree → a condensed cluster hierarchy. The distance/MST stage is roughly O(n²) in time and memory, so 6.3M wallets × 26 features is far past what a laptop holds, and even aggressive subsampling distorts the density estimates HDBSCAN depends on. A Colab A100 (40 GB, GPU-accelerated via RAPIDS cuML) makes the full-population fit feasible.


## Load model-ready features

Read wallet_features_v5_model.parquet — the standardized, address-less matrix from notebook 02.


In [ ]:
import time, numpy as np, pandas as pd, cupy as cp
from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN

def timer(label):
    class T:
        def __enter__(s): s.t=time.perf_counter(); return s
        def __exit__(s,*a): print(f"⏱  {label}: {time.perf_counter()-s.t:.2f}s")
    return T()

FEAT_PATH = '/content/drive/MyDrive/share/wallet_features_v5_model.parquet'
LUT_PATH  = '/content/drive/MyDrive/share/wallet_label_lut.parquet'

with timer("load"):
    full = pd.read_parquet(FEAT_PATH)
    lut  = pd.read_parquet(LUT_PATH)

# --- alignment guard: positional join is only valid if rows line up 1:1 ---
assert len(full) == len(lut), f"row count mismatch: feats={len(full)} lut={len(lut)}"
print(f"rows aligned: {len(full):,}")

## Fit HDBSCAN (cuML, A100)

Density clustering over all 6.3M wallets; yields a cluster label and membership strength per wallet.


In [ ]:
# --- 10% sample (same seed = same rows as all prior diagnostics) ---
sample = full.sample(frac=0.10, random_state=42)
sample_pos = full.index.get_indexer(sample.index)   # positional rows of the sample, for writeback
X = sample.values.astype('float32')
assert not np.isnan(X).any(), "NaN in X — fix before UMAP"
print(f"sample: {sample.shape}")

# --- UMAP 10D (locked clustering params: n_neighbors=22) ---
with timer("UMAP 10D"):
    emb10 = cp.asnumpy(UMAP(n_components=10, n_neighbors=22, min_dist=0.0,
                            metric='euclidean', random_state=42).fit_transform(X))

## Clusters → archetype families

Inspect each cluster's feature profile and map cluster id → family name (cluster_archetypes.json).


In [ ]:
{
  "0":  {"name": "token_only_defi_user",         "confidence": "high",   "validated": false},
  "1":  {"name": "token_only_transferrer",       "confidence": "high",   "validated": false},
  "2":  {"name": "phishing_drain_target",        "confidence": "high",   "validated": true},
  "3":  {"name": "meme_token_degen_trader",      "confidence": "high",   "validated": false},
  "4":  {"name": "mainstream_net_sender",        "confidence": "low",    "validated": false},
  "5":  {"name": "phishing_drain_funnel",        "confidence": "high",   "validated": true},
  "6":  {"name": "simple_plain_eth_eoa",         "confidence": "medium", "validated": false},
  "7":  {"name": "minter",                       "confidence": "high",   "validated": false},
  "8":  {"name": "multi_router_defi_bot",        "confidence": "high",   "validated": true},
  "9":  {"name": "pure_receiver_deposit_addr",   "confidence": "high",   "validated": false},
  "10": {"name": "pure_receiver_gas_aware",      "confidence": "high",   "validated": false},
  "11": {"name": "mainstream_holder_transferrer","confidence": "low",    "validated": false},
  "12": {"name": "plain_eth_holder_funded",      "confidence": "medium", "validated": false},
  "13": {"name": "token_active_holder_funded",   "confidence": "medium", "validated": false},
  "14": {"name": "mainstream_variant",           "confidence": "low",    "validated": false},
  "15": {"name": "occasional_transferrer",       "confidence": "low",    "validated": false},
  "16": {"name": "dex_aggregator_user",          "confidence": "high",   "validated": true},
  "17": {"name": "active_dex_trader",            "confidence": "high",   "validated": false},
  "-1": {"name": "noise_unclustered",            "confidence": "n/a",    "validated": false},
  "-2": {"name": "not_in_sample",                "confidence": "n/a",    "validated": false}
}

## XGBoost classifier for unclustered wallets

Train on the HDBSCAN labels so noise / brand-new wallets get a predicted cluster (xgb_cluster_clf.joblib, scaler_v5.joblib).

In [ ]:
import pandas as pd, numpy as np, xgboost as xgb, joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

feats = pd.read_parquet('/content/drive/MyDrive/share/wallet_features_v5_model.parquet')  # z-scored, no address
lut   = pd.read_parquet('/content/drive/MyDrive/share/wallet_label_lut.parquet')
assert len(feats) == len(lut), "row mismatch — alignment broken"

# y_all = lut['cluster'].values
# train_mask = y_all >= 0                      # 0..17 only; drop -1 noise and -2 unsampled
# X = feats.values[train_mask].astype('float32')
# y = y_all[train_mask].astype('int32')
# print(f"training rows: {len(y):,} across {len(np.unique(y))} classes")

# --- in Cell 1, replace the masking + class setup ---
y_all = lut['cluster'].values
train_mask = y_all >= -1                      # KEEP -1 (noise) this time; still drop -2 (unsampled)
X = feats.values[train_mask].astype('float32')
y_raw = y_all[train_mask]

NOISE_CLASS = 18                              # remap -1 -> 18 so labels are 0..18 contiguous
y = np.where(y_raw == -1, NOISE_CLASS, y_raw).astype('int32')
print(f"training rows: {len(y):,} | classes: {sorted(np.unique(y))} (18 = noise)")
# ...rest identical, num_class=19...

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# class weights -> sample weights (handles the c6=70k vs c7=5.8k imbalance)
cls, cnt = np.unique(ytr, return_counts=True)
w = {c: len(ytr)/(len(cls)*n) for c, n in zip(cls, cnt)}
wtr = np.array([w[c] for c in ytr], dtype='float32')

clf = xgb.XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    objective='multi:softprob', num_class=len(cls),
    tree_method='hist', device='cuda',        # A100 GPU
    eval_metric='mlogloss', early_stopping_rounds=30,
)
clf.fit(Xtr, ytr, sample_weight=wtr, eval_set=[(Xte, yte)], verbose=20)

pred = clf.predict(Xte)
print(classification_report(yte, pred, digits=3))
joblib.dump((clf, list(feats.columns)), '/content/drive/MyDrive/share/xgb_cluster_clf.joblib')
print("saved -> xgb_cluster_clf.joblib")

# feature importance — doubles as a report artifact
imp = pd.Series(clf.feature_importances_, index=feats.columns).sort_values(ascending=False)
print("\ntop features:\n", imp.head(12).round(3).to_string())

## XGBoost classifier for unclustered wallets - Inference/

In [ ]:
import joblib
# after: scaler = StandardScaler(); df[feats] = scaler.fit_transform(df[feats])
joblib.dump(scaler, '/content/drive/MyDrive/share/scaler_v5.joblib')

import pandas as pd, numpy as np, joblib, matplotlib.pyplot as plt

# --- load ---
feats = pd.read_parquet('/content/drive/MyDrive/share/wallet_features_v5_model.parquet')
lut   = pd.read_parquet('/content/drive/MyDrive/share/wallet_label_lut.parquet')
clf, cols = joblib.load('/content/drive/MyDrive/share/xgb_cluster_clf.joblib')
assert len(feats) == len(lut), "row mismatch — alignment broken"
assert list(feats.columns) == cols, "feature order mismatch"

NOISE_CLASS = 18                                  # class 18 = noise (remapped from -1 in training)
orig = lut['cluster'].values

# --- predict all 6.3M ---
X = feats.values.astype('float32')
proba = clf.predict_proba(X)
pred  = proba.argmax(1).astype('int16')
conf  = proba.max(1).astype('float32')
pred_is_noise = pred == NOISE_CLASS

# --- inspect confidence BEFORE committing: pick TAU from the valley ---
fill = orig < 0                                   # the 90% we're about to fill
real_pred = fill & ~pred_is_noise                 # filled rows predicted as a real cluster
plt.figure(figsize=(8,4))
plt.hist(conf[real_pred], bins=60, color='steelblue')
plt.axvline(0.5, color='red', ls='--', label='TAU=0.5 (default)')
plt.title("XGB confidence on filled rows predicted to a real cluster")
plt.xlabel("max class probability"); plt.ylabel("count"); plt.legend(); plt.show()

# --- three-way assignment ---
TAU = 0.5                                          # <-- set from the histogram valley above
low_conf = conf < TAU
final = np.where(pred_is_noise, -1, pred)                          # model says noise -> -1
final = np.where((~pred_is_noise) & low_conf, -1, final).astype('int16')  # too unsure -> -1

# --- write into LUT: keep HDBSCAN labels, fill only the -2 rows ---
lut['cluster_xgb']  = np.where(orig >= 0, orig, final).astype('int16')
lut['xgb_conf']     = conf
lut['label_source'] = np.select(
    [orig >= 0, (orig < 0) & (final >= 0)],
    ['hdbscan', 'xgb_predicted'],
    default='xgb_noise')                                            # model-noise or low-conf

# --- diagnostics ---
print("=== filled-row outcomes (the 90%) ===")
print(f"  predicted real cluster (kept):  {(fill & (final>=0)).sum():>9,}")
print(f"  predicted noise -> -1:          {(fill & pred_is_noise).sum():>9,}  ({pred_is_noise[fill].mean()*100:.1f}%)")
print(f"  low-conf (<{TAU}) -> -1:          {(fill & ~pred_is_noise & low_conf).sum():>9,}  ({((~pred_is_noise)&low_conf)[fill].mean()*100:.1f}%)")
print(f"  total -> -1 on filled rows:     {(final[fill]==-1).mean()*100:.1f}%   (HDBSCAN sample was 37.9%)")
print(f"\n  mean conf on kept filled rows:  {conf[fill & (final>=0)].mean():.3f}")

print("\n=== final cluster_xgb distribution ===")
print(pd.Series(lut['cluster_xgb']).value_counts().sort_index().to_string())

print("\n=== label_source breakdown ===")
print(lut['label_source'].value_counts().to_string())

lut.to_parquet('/content/drive/MyDrive/share/wallet_label_lut.parquet', index=False)
print("\nsaved -> wallet_label_lut.parquet")

## Build the wallet lookup table

Join the HDBSCAN and XGBoost results into wallet_lut.parquet (address → effective cluster + label_source) — the table the backend serves.


In [ ]:
import pandas as pd, numpy as np, joblib

feats = pd.read_parquet('/content/drive/MyDrive/share/wallet_features_v5_model.parquet')
lut   = pd.read_parquet('/content/drive/MyDrive/share/wallet_label_lut.parquet')
clf, cols = joblib.load('/content/drive/MyDrive/share/xgb_cluster_clf.joblib')
assert list(feats.columns) == cols, "feature order mismatch"

X = feats.values.astype('float32')
proba = clf.predict_proba(X)                  # all 6.3M
pred  = proba.argmax(1).astype('int16')
conf  = proba.max(1).astype('float32')

# keep HDBSCAN's original labels where they exist; XGBoost only fills the -2 (unsampled) rows
orig = lut['cluster'].values
lut['cluster_xgb']  = np.where(orig >= 0, orig, pred)          # final label per wallet
lut['xgb_conf']     = conf                                     # prediction confidence
lut['label_source'] = np.where(orig >= 0, 'hdbscan', 'xgb_predicted')

print("final label distribution (cluster_xgb):")
print(pd.Series(lut['cluster_xgb']).value_counts().sort_index().to_string())
print(f"\nmean XGB confidence on filled rows: {conf[orig<0].mean():.3f}")
print(f"low-confidence (<0.5) filled rows: {(conf[orig<0]<0.5).mean()*100:.1f}%")

lut.to_parquet('/content/drive/MyDrive/share/wallet_label_lut.parquet', index=False)
print("saved -> wallet_label_lut.parquet")